In [ ]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"

!pip install -q -U triton

# 3. Install Unsloth (Kaggle version)
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 9.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.9.0+cu126 requires triton==3.5.0; platform_system == "Linux", but you have triton 3.6.0 which is incompatible.
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-jado69nm/unsloth_5353e2651a4d4dc2be65c27f941e3480
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-jado69nm/unsloth_5353e2651a4d4dc2be65c27f941e3480
  Resolved https://github.com/unslothai/unsloth.git to commit 050240b27a0c0cba0f33a1720e14dde8f48a6eb2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import pandas as pd
# Load the CSV
column_names = ['target', 'ids', 'date', 'flag', 'user', 'text']
uncleaned_dataset = pd.read_csv("/kaggle/input/datasets/majeedabdul03/datasets-term-paper/uncleaned_data.csv", names=column_names, encoding='latin-1', engine='python')

# 1. Drop the neutral tweets (target == 2) 
uncleaned_dataset = uncleaned_dataset[uncleaned_dataset['target'] != 2].copy()

# 2. Map the targets to match your cleaned dataset's 0 and 1
sentiment_mapping = {
    0: "0",  # Negative
    4: "1"   # Positive becomes 1
}
uncleaned_dataset['target'] = uncleaned_dataset['target'].map(sentiment_mapping)

# 3. Isolate and rename 'target' to 'polarity'
uncleaned_dataset = uncleaned_dataset[['text', 'target']].copy()

uncleaned_dataset = uncleaned_dataset.rename(columns={
    'target': 'polarity'
})



In [ ]:

# Randomly sample exactly 5000 rows
df_full=uncleaned_dataset
df_test = df_full.sample(n=10000, random_state=554)

# Reset the index so the rows are numbered 0 to 999 cleanly
df_test = df_test.reset_index(drop=True)

print(f"Test set successfully created with {len(df_test)} rows.")

# Optional: Save this test set to a new CSV so you have a permanent copy of it
df_test.to_csv("my_1000_row_test_set.csv", index=False)

Test set successfully created with 10000 rows.


In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto-detects (Float16 for T4)
load_in_4bit = True # Essential for Colab

# Load the base model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-2-7b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.87G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/948 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-2-7b-bnb-4bit as a legacy tokenizer.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA rank - controls the adapter size
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # 0 is optimized for Unsloth
    bias = "none",    # "none" is optimized for Unsloth
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
)

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
from datasets import Dataset
df_train=uncleaned_dataset
unclean_dataset=Dataset.from_pandas(df_train)

#standard instruction format
prompt_template = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["text"]
    outputs      = examples["polarity"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        # Must add the EOS token to prevent endless generation
        text = prompt_template.format(instruction, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

# Map the format dataset
unclean_dataset = unclean_dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/1600000 [00:00<?, ? examples/s]

In [ ]:
import torch._dynamo
torch._dynamo.disable()
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = unclean_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 800, # Increased to 800 for better fitting
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        # UPDATE 1: Save intermediate training checkpoints to Kaggle's working directory
        output_dir = "/kaggle/working/outputs_unclean", 
        average_tokens_across_devices = False,
    ),
)

#  training
trainer.train()


adapter_unclean = "/kaggle/working/lora_adapter_uncleaned"

# Save the final weights
model.save_pretrained(adapter_unclean)
tokenizer.save_pretrained(adapter_unclean)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/1600000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,600,000 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 39,976,960 of 6,778,392,576 (0.59% trained)


Step,Training Loss
1,3.791063
2,3.418490
3,3.566270
4,3.439480
5,3.250990
6,3.052754
7,2.735797
8,2.593714
9,2.210100
10,2.086180


('/kaggle/working/lora_adapter_uncleaned/tokenizer_config.json',
 '/kaggle/working/lora_adapter_uncleaned/tokenizer.json')

In [8]:
import gc
del trainer, model
torch.cuda.empty_cache()
gc.collect()

44848

In [9]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-2-7b-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16, lora_dropout = 0, bias = "none", use_gradient_checkpointing = "unsloth", random_state = 3407,
)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-2-7b-bnb-4bit as a legacy tokenizer.


In [10]:

df_train1=pd.read_csv('/kaggle/input/datasets/majeedabdul03/datasets-term-paper/cleaned_train.csv')
clean_dataset = Dataset.from_pandas(df_train1)
clean_dataset = clean_dataset.map(formatting_prompts_func, batched = True)




Map:   0%|          | 0/500000 [00:00<?, ? examples/s]

In [ ]:
import torch._dynamo
torch._dynamo.disable()
from trl import SFTTrainer
from transformers import TrainingArguments

trainer1 = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = clean_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 800, # Increased this for full training runs
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        # UPDATE 1: Save intermediate training checkpoints to Kaggle's working directory
        output_dir = "/kaggle/working/outputs_clean", 
        average_tokens_across_devices = False,
    ),
)

#  training
trainer1.train()

# final adapter save location
adapter_clean = "/kaggle/working/lora_adapter_cleaned"

# Save the final weights
model.save_pretrained(adapter_clean)
tokenizer.save_pretrained(adapter_clean)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/500000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500,000 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 39,976,960 of 6,778,392,576 (0.59% trained)


Step,Training Loss
1,4.994014
2,3.554231
3,3.604346
4,3.166108
5,3.533638
6,3.407281
7,2.204293
8,3.034260
9,2.737329
10,2.435137


('/kaggle/working/lora_adapter_cleaned/tokenizer_config.json',
 '/kaggle/working/lora_adapter_cleaned/tokenizer.json')

In [ ]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Load the test set
df_test = pd.read_csv("my_1000_row_test_set.csv")


df_test["polarity"] = df_test["polarity"].astype(str)

#  fast inference mode
FastLanguageModel.for_inference(model)

adapters = ["lora_adapter_cleaned", "lora_adapter_uncleaned"]
results = {}

for adapter in adapters:
    print(f"\n--- Evaluating {adapter} ---")

    # Load the specific adapter weights
    model.load_adapter(adapter, adapter_name=adapter)

    model.set_adapter(adapter)

    predictions = []
    true_labels = df_test["polarity"].tolist()

    for text in tqdm(df_test["text"]):
        prompt = f"""Analyze the text below and classify its sentiment.
Reply with exactly '0' for Negative or '1' for Positive.

### Text:
{text}

### Polarity:
"""
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=2, use_cache=True)

        prediction = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

        if prediction not in ["0", "1"]:
            prediction = "0"

        predictions.append(prediction)

    # Calculate Metrics
    acc = accuracy_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions, pos_label="1")

    results[adapter] = {"Accuracy": acc, "F1 Score": f1}

    print(classification_report(true_labels, predictions))

# Final Comparison
print("\n=== FINAL RESULTS ===")
print(pd.DataFrame(results).T)


--- Evaluating lora_adapter_cleaned ---


  0%|          | 0/10000 [00:00<?, ?it/s]--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/tra

              precision    recall  f1-score   support

           0       0.51      0.68      0.58      5008
           1       0.52      0.34      0.41      4992

    accuracy                           0.51     10000
   macro avg       0.51      0.51      0.50     10000
weighted avg       0.51      0.51      0.50     10000


--- Evaluating lora_adapter_uncleaned ---


100%|██████████| 10000/10000 [47:23<00:00,  3.52it/s]

              precision    recall  f1-score   support

           0       0.54      0.80      0.65      5008
           1       0.62      0.32      0.42      4992

    accuracy                           0.56     10000
   macro avg       0.58      0.56      0.53     10000
weighted avg       0.58      0.56      0.53     10000


=== FINAL RESULTS ===
                        Accuracy  F1 Score
lora_adapter_cleaned      0.5118  0.412798
lora_adapter_uncleaned    0.5607  0.419146
